<a href="https://colab.research.google.com/github/GregKyri/ToDrinQ_WP6/blob/main/dwtpcost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Cost estimations using Cost calculator and EPA tool***

In [1]:
import math as mh
import pandas as pd
import numpy as np

def dwtpcost(Treatment,dailyflow, Epump,stor_area, coagtype,Ecoag,Efloc,Eclar,amountofcoagday,Etot,Eback,Eomp,Edis,Cldaily,ammdaily,softenergy,lime_dosage_kg_per_day,ufenergy, nomenergy, nomchem,condenergy,condchem,aerarea,aerenergy):

    """
    Compute CAPEX and OPEX for each treatment step in a drinking water plant.

    Parameters
    ----------
    treatment_df : pd.DataFrame
        Columns required:
            - 'Treatment Steps'
            - 'Treatment Technologies'
    dailyflow : float | list | np.ndarray
        Flow in m³/day.
    coagtype : {'Alum', 'Iron'}, default 'Alum'
        Determines coagulant type used in Coagulation step.
    Energy variables:
        Energy consumption per m³ for specific steps (e.g., Ecoag, Efloc, etc.).

    Returns
    -------
    step_costs : dict
        Per-step cost breakdown.
    totals : dict
        Total cost across all steps.
    """

   # Flow in m³/day
    Q = float(dailyflow[0] if isinstance(dailyflow, (list, tuple, np.ndarray)) else dailyflow)
    Qyear=Q*365 # yearly production of water
    Q=Q/24 # Flow in m³/h

      # --- Step-specific cost definitions ---
    # Format: 'Step': {'type': 'linear' or 'poly', 'construction': [...], 'maintenance': [...]}


    # User input: select country
    country = input("Please clarify the country for which the construction costs are calculated (Greece or the Netherlands)") # Change to "Netherlands" if needed

    # Set parameters based on country
    if country.lower() == "greece":
      interest_rate = 0.07  # 7% nominal interest rate
      inflation_rate = 0.036  # 3.6% current inflation (June 2025)
    elif country.lower() == "netherlands":
      interest_rate = 0.05  # 5% nominal interest rate
      inflation_rate = 0.031  # 3.1% current inflation (June 2025)
    else:
      raise ValueError("Country must be 'Greece' or 'Netherlands'")

    # Constants
    print('We assume that the lifespane of the drinking water treatment plant is 30 years')
    n = 30  # years

    # Capital Recovery Factor (CRF)
    r_real = (1 + interest_rate) / (1 + inflation_rate) - 1
    if r_real <= -1:
        return np.nan  # Avoid division by zero or invalid cases
    crf = (r_real * (1 + r_real)**n) / ((1 + r_real)**n - 1)


    cost_models = {
        "Intake pumping station": {
            'type': 'poly',
            'construction': [-0.20849, 1446.4031,261110.8619],
            'maintenance': [-0.0042948, 29.7959, 8078.884],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Screens": {
            'type': 'linear',
            'construction': [573.32, 238518.2],
            'maintenance': [19.27, 6322.11],
            'materials': [0],
            'energy': [0],
            'chemicals': [0]
        },
        "Microsieves": {
            'type': 'linear',
            'construction': [1023.59, 278408.75],
            'maintenance': [22.365, 7433.23],
            'materials': [0],
            'energy': [0],
            'chemicals': [0]
        },
        "Trickling Filtration": {
            'type': 'linear',
            'construction': [0, 0],
            'maintenance': [0, 0],
            'materials': [0],
            'energy': [0],
            'chemicals': [0]
        },
        "Infiltration": {
            'type': 'linear',
            'construction': [1166.8281, 311764.583],
            'maintenance': [23.92, 18721.174],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Coagulation with aluminium coagulant": {
            'type': 'linear',
            'construction': [12.949, 29556.185],
            'maintenance': [0.676, 3342.833],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.35]
        },
        "Coagulation with iron coagulant": {
            'type': 'linear',
            'construction': [47.252, 30596.716],
            'maintenance': [2.467, 3397.149],
            'materials': [0, 0],
            'energy': [0.14],
            'chemicals': [0.535]
        },
        "Static mixing": {
            'type': 'poly',
            'construction': [-0.14903, 2605.204, 76836.342],
            'maintenance': [-0.0032191, 56.272, 3459.67],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Dynamic mixing": {
            'type': 'poly',
            'construction': [-0.14903, 2605.204, 76836.342],
            'maintenance': [-0.0032191, 56.272, 3459.67],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Dissolved Air Flotation": {
            'type': 'poly',
            'construction': [-0.43731, 2712.45, 1595592.14],
            'maintenance': [-0.00879, 54.52, 35671.402],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Sludge Blanket Clarification": {
            'type': 'poly',
            'construction': [0.1212, 2333.45, 132452.56],     # a*Q^3 + b*Q^2 + c*Q + d
            'maintenance': [-0.0054497, 66.425, 7509.7],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Horizontal Settling": {
            'type': 'poly',
            'construction': [0.2812, 2669.91, 165855.156],     # a*Q^3 + b*Q^2 + c*Q + d
            'maintenance': [-0.0054497, 66.425, 7509.7],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Titled-plate Settling": {
            'type': 'poly',
            'construction': [0.30109, 3669.91, 265732.356],     # a*Q^3 + b*Q^2 + c*Q + d
            'maintenance': [-0.018268, 119.896, 33411.254],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Pellet Softening": {
            'type': 'poly',
            'construction': [-1.0446, 6855.523, 1138508.81],
            'maintenance': [-0.018268, 119.896, 33411.254],
            'materials': [0, 2.0257,0],
            'energy': [0.14],
            'chemicals': [0.16]
        },
        "NF/RO": {
            'type': 'poly',
            'construction': [-0.99643, 10118.208, 251325.12],
            'maintenance': [-0.0254, 258.01, 19908.79],
            'materials': [0, 79.235, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "UF": {
            'type': 'poly',
            'construction': [0.0014526, -6.7762, 18774.38, 729603.436],
            'maintenance': [0.000003, -0.16907, 468.421, 24953.61],
            'materials': [0, 172.851, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "MF": {
            'type': 'poly',
            'construction': [0.0014526, -6.7762, 18774.38, 729603.436],
            'maintenance': [0.000003, -0.16907, 468.421, 24953.61],
            'materials': [0, 172.851, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
        "Single Media Filtration": {
            'type': 'poly',
            'construction': [-0.61497, 4266.467, 875229.97],
            'maintenance': [-0.11561, 80.21, 21854.323],
            'materials': [0, 9.643, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Dual Media Filtration": {
            'type': 'poly',
            'construction': [-0.61497, 4266.467, 875229.97],
            'maintenance': [-0.11561, 80.21, 21854.323],
            'materials': [0, 12.443, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Moving Bed Filtration": {
            'type': 'poly',
            'construction': [-0.61497, 4266.467, 875229.97],
            'maintenance': [-0.11561, 80.21, 21854.323],
            'materials': [0, 9.643, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Pressurized Filtration Vessel": {
            'type': 'poly',
            'construction': [0.00029578, -1.415, 4472.463, 137349.86],
            'maintenance': [0.000006, -0.0292, 92.13, 8229.41],
            'materials': [0, 6.1147, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Granular Activated Carbon Filtration (GAC)": {
            'type': 'poly',
            'construction': [-0.4764, 4554.89, 992898.38],
            'maintenance': [0.0095757, 91.5534, 92.13, 23557.26],
            'materials': [0, 172851, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Powdered Activated Carbon Dosing (PAC)": {
            'type': 'linear',
            'construction': [193.36, 71460.17],
            'maintenance': [4.911, 4065.09],
            'materials': [46.607, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Ozonation": {
            'type': 'linear',
            'construction': [539.12, 81496.51],
            'maintenance': [13.80, 44786.31],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "MP-UV/H2O2": {
            'type': 'linear',
            'construction': [55.56, 30848.84],
            'maintenance': [2.9, 3410.31],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.375]
        },
         "Chlorination": {
            'type': 'linear',
            'construction': [49.6937, 34631.7445],
            'maintenance': [2.594, 3607.7771],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.51]
        },
          "Chloramination": {
            'type': 'linear',
            'construction': [50.78, 30703.78],
            'maintenance': [2.65, 3402.73],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.51, 0.265]
        },
         "LP-UV": {
            'type': 'poly',
            'construction': [-0.045486, 330.87, 91257.68],
            'maintenance': [0, 8.7349, 7809.2029],
            'materials': [0, 16.8117, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Cascades": {
            'type': 'poly',
            'construction': [-0.048535, 408.35, 68577.34],
            'maintenance': [0.000747, 6.2885, 1686.09],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Spray Aeration": {
            'type': 'poly',
            'construction': [-0.13255, 1383.7884, 119398.54],
            'maintenance': [0.003, 31, 4924.53],
            'materials': [0, 16.8117, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Plate Aeration": {
            'type': 'poly',
            'construction': [-0.13255, 1383.7884, 119398.54],
            'maintenance': [0.003, 31, 4924.53],
            'materials': [0, 16.8117, 0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Stripping Tower": {
            'type': 'poly',
            'construction': [0, 721.7644, 45357.8087],
            'maintenance': [-0.048535, 408.3471, 68577.34],
            'materials': [0, 0, 294],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Dosing of Chemicals": {
            'type': 'linear',
            'construction': [42.8475, 30352.5875],
            'maintenance': [2.2366, 3384.4051],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.3]
        },
         "pH Correction": {
            'type': 'linear',
            'construction': [42.8475, 30352.5875],
            'maintenance': [2.2366, 3384.4051],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.16]
        },
         "Remineralisation": {
            'type': 'linear',
            'construction': [42.8475, 30352.5875],
            'maintenance': [2.2366, 3384.4051],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.1]
        },
         "Enhanced Coagulation": {
            'type': 'linear',
            'construction': [12.949, 29556.185],
            'maintenance': [0.676, 3342.833],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0.35]
        },
         "Anionic IEX": {
            'type': 'linear',
            'construction': [1032.2851, 714879.942],
            'maintenance': [149.918, 13324.0576],
            'materials': [7.8963, 11447.326],
            'energy': [0.14],
            'chemicals': [0.235]
        },
         "Cationic IEX": {
            'type': 'linear',
            'construction': [1237.4049, 742039.8794],
            'maintenance': [437.0074, 38724.247],
            'materials': [11.0374, 13826.9393],
            'energy': [0.14],
            'chemicals': [0.235]
        },
         "Boosting Pumps": {
            'type': 'linear',
            'construction': [601.9495, 183829.3755],
            'maintenance': [16.0119, 7589.8614],
            'materials': [0],
            'energy': [0.14],
            'chemicals': [0]
        },
         "Clean Water Storage Tank": {
            'type': 'poly',
            'construction': [-0.35174, 3444.5561, 234797.7386],
            'maintenance': [-0.037988, 37.2012, 3435.8156],
            'materials': [0],
            'energy': [0],
            'chemicals': [0]
        },
    }

    def _eval(coeffs):
        return crf*float(np.polyval(coeffs, Q)) if isinstance(coeffs, (list, tuple, np.ndarray)) else crf*float(coeffs)

    totals = {
        "construction": 0.0,
        "maintenance": 0.0,
        "materials": 0.0,
        "energy": 0.0,
        "chemicals": 0.0
    }
    step_costs = {}

    step_energy_vars = {
        "Intake pumping station": Epump*365,
        "Coagulation": Ecoag*365,
        "Flocculation": Efloc*365,
        "Floc Removal": Eclar*365,
        "Sand Filtration": (Etot + Eback)*365,
        "OMP removal": Eomp*365,
        "Disinfection": Edis*365,
        "Ultrafiltration": ufenergy*365,
        "NOM removal": nomenergy*365,
        "Softening": softenergy*365,
        "Aeration": aerenergy*365,
        "Conditioning":condenergy*365
    }

    for _, row in Treatment.iterrows():
        step = str(row["Treatment Steps"]).strip()
        tech = str(row["Treatment Technologies"]).strip()

        if step.lower() == "coagulation":
            tech = "Coagulation with aluminium coagulant" if coagtype.lower().startswith("alum") else "Coagulation with iron coagulant"

        model = cost_models.get(tech)
        if model is None:
            print(f"⚠️  No cost model found for: {tech}")
            continue

        c_con = _eval(model.get("construction", 0))
        c_mai = _eval(model.get("maintenance", 0))
        c_mat = _eval(model.get("materials", 0))

        # Energy cost
        energy_var = step_energy_vars.get(step, 0.0)
        c_energy = _eval(model.get("energy", 0)) * energy_var

        # Chemical cost
        c_chem = 0.0
        if tech in ["Coagulation with aluminium coagulant", "Coagulation with iron coagulant"]:
            c_chem = _eval(model.get("chemicals", 0)) * amountofcoagday * 365
        elif tech == "Pellet Softening":
            c_chem = _eval(model.get("chemicals", 0)) * lime_dosage_kg_per_day * 365
        elif tech == "Cationic IEX":
            c_chem = _eval(model.get("chemicals", 0)) * lime_dosage_kg_per_day * 365
        elif tech == "Anionic IEX":
            c_chem = _eval(model.get("chemicals", 0)) * nomchem * 365
        elif tech == "pH Correction":
            c_chem = _eval(model.get("chemicals", 0)) * condchem  * 365
        elif tech == "Dosing of Chemicals":
            c_chem = _eval(model.get("chemicals", 0)) * condchem * 365
        elif tech == "Remineralisation":
            c_chem = _eval(model.get("chemicals", 0)) * condchem * 365
        elif tech == "Chlorination":
            c_chem = _eval(model.get("chemicals", 0)) * Cldaily * 365
        elif tech == "Chloramination":
            chem_vals = [0.51, 0.265]
            #if isinstance(chem_vals, list) and len(chem_vals) == 2:
            c_chem = chem_vals[0] * Cldaily * 365 + chem_vals[1] * ammdaily * 365
        else:
            c_chem = _eval(model.get("chemicals", 0))

        step_costs[tech] = dict(
            construction=c_con,
            maintenance=c_mai,
            materials=c_mat,
            energy=c_energy,
            chemicals=c_chem,
        )

        print(f"\n🔹 Step: {tech}")
        print(f"   - Annual Construction Cost: €{c_con:,.2f}")
        print(f"   - Yearly Maintenance Cost:  €{c_mai:,.2f}")
        print(f"   - Yearly Materials Cost:    €{c_mat:,.2f}")
        print(f"   - Yearly Energy Costs:       €{c_energy:,.2f}")
        print(f"   - Yearly Chemicals Costs:    €{c_chem:,.2f}")
        print(f"   ➤ Annual Total Costs:        €{c_con + c_mai + c_mat + c_energy + c_chem:,.2f}")
        print(f"   ➤ Cost per Cubic Meter Produced:        {(c_con + c_mai + c_mat + c_energy + c_chem)/Q:,.2f}")

        for k, v in step_costs[tech].items():
            totals[k] += step_costs[tech][k]

    totals["grand_total"] = sum(totals.values())
    totals["Yearly cost per cubic meter"] = totals["grand_total"]/Q
    print("\n============================")
    print("🧾 Total Yearly Cost Summary (€)")
    print("============================")
    for k, v in totals.items():
        if k != "grand_total" and k != "Yearly cost per cubic meter":
            print(f"   {k.capitalize():<12}: €{v:,.2f}")
    print(f"   {'Annual Total Cost':<12}: €{totals['grand_total']:,.2f}")
    print(f"   {'Cost per cubic meter produced':<12}: {totals['Yearly cost per cubic meter']:,.2f}€/m^3")
    print("============================\n")
    return step_costs, totals